Labeling Play Videos Outcomes

In [1]:
from pathlib import Path
import pandas as pd

# <<< EDIT THESE >>>
video_dir = Path("data/plays_outcome_labeling")         # folder containing your play videos
label_file = Path("data/play_labels.csv")  # CSV to store labels

# Your 13 predefined outcomes + 1 catch-all
LABELS = [
    "Run_TFL",        # examples — replace with your real categories
    "Run_0-10",
    "Run_10+",
    "Pass_INC_Negative",
    "Pass_INC_0-20",
    "Pass_INC_20+",
    "Pass_INT_Negative",
    "Pass_INT_0-20",
    "Pass_INT_20+",
    "Pass_COM_Negative",
    "Pass_COM_0-20",
    "Pass_COM_20+",
    "Sack",
    "Other"  # e.g. punts, FGs, weird stuff
]

SIDELINE_SUFFIX = "_Sideline.mp4"   # your naming convention

In [2]:
from IPython.display import Video, display, clear_output
import ipywidgets as widgets

assert video_dir.exists(), f"Video directory not found: {video_dir.resolve()}"

# All mp4s in the folder
all_videos = sorted(video_dir.glob("*.mp4"))

def is_sideline_view(path):
    """Return True if this file is a sideline view based on suffix."""
    return path.name.endswith(SIDELINE_SUFFIX) or path.name.lower().endswith("_sideline.mp4")

# Filter to sideline-only
video_files = [vf for vf in all_videos if is_sideline_view(vf)]

print(f"Found {len(all_videos)} total videos.")
print(f"Using {len(video_files)} sideline videos (ending with {SIDELINE_SUFFIX}) for labeling.")

if not video_files:
    raise RuntimeError(
        f"No sideline videos found ending with {SIDELINE_SUFFIX}. "
        "Double-check filenames and SIDELINE_SUFFIX."
    )

# Load existing labels if present
if label_file.exists():
    df_labels = pd.read_csv(label_file)
    labeled = dict(zip(df_labels["video_file"], df_labels["label"]))
else:
    df_labels = pd.DataFrame(columns=["video_file", "label"])
    labeled = {}

# Start at first unlabeled sideline video
current_index = 0
while current_index < len(video_files) and video_files[current_index].name in labeled:
    current_index += 1

# Widgets
label_dropdown = widgets.Dropdown(
    options=LABELS,
    description="Outcome:",
    value=LABELS[0],
    layout=widgets.Layout(width="300px")
)

save_button = widgets.Button(
    description="Save label",
    button_style="success",
    layout=widgets.Layout(width="120px")
)

skip_button = widgets.Button(
    description="Skip",
    button_style="warning",
    layout=widgets.Layout(width="80px")
)

status_label = widgets.Label()

controls = widgets.VBox([
    status_label,
    label_dropdown,
    widgets.HBox([save_button, skip_button])
])


def save_labels_to_csv():
    """Write the labeled dict to CSV."""
    global df_labels
    df_labels = pd.DataFrame(
        [{"video_file": name, "label": lab} for name, lab in labeled.items()]
    )
    df_labels.to_csv(label_file, index=False)


def show_current_video():
    """Display the current video and controls."""
    clear_output(wait=True)
    display(controls)

    if current_index >= len(video_files):
        status_label.value = (
            f"All sideline videos labeled! Total labeled: {len(labeled)}"
        )
        return

    vf = video_files[current_index]
    status_label.value = (
        f"Sideline video {current_index + 1}/{len(video_files)}: {vf.name}"
    )
    display(Video(str(vf), embed=True))


def on_save_clicked(b):
    global current_index

    if current_index >= len(video_files):
        return

    vf = video_files[current_index]
    chosen_label = label_dropdown.value

    # Save in memory
    labeled[vf.name] = chosen_label
    # Persist to CSV
    save_labels_to_csv()

    current_index += 1
    show_current_video()


def on_skip_clicked(b):
    global current_index
    if current_index >= len(video_files):
        return
    current_index += 1
    show_current_video()


save_button.on_click(on_save_clicked)
skip_button.on_click(on_skip_clicked)

# Start UI
show_current_video()